In [8]:
#%% HEADER

# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models

from destruction_utilities import *
from numpy import random
from matplotlib import pyplot
from keras import callbacks, preprocessing
from os import path

#%% FUNCTIONS

def display_history(history:dict, stats:list=['accuracy', 'loss']) -> None:
    '''Displays model training history'''
    fig, axs = pyplot.subplots(nrows=1, ncols=2, figsize=(10, 5))
    for ax, stat in zip(axs.ravel(), stats):
        ax.plot(history[stat])
        ax.plot(history[f'val_{stat}'])
        ax.set_title(f'Training {stat}', fontsize=15)
        ax.set_ylabel('Accuracy')
        ax.set_xlabel('Epoch')
        ax.legend(['Training sample', 'Validation sample'], frameon=False)
    pyplot.tight_layout(pad=2.0)
    pyplot.show()

#%% DECLARATION
CITY = 'daraa'

#%% READS DATA
# Reads images (subset)
images = search_data(pattern(city=CITY, type='image'))
images = np.array(list(map(read_raster, images)))
images = tile_sequences(images, tile_size=(128, 128))

# Reads labels (subset)
labels = search_data(pattern(city=CITY, type='label'))
labels = np.array(list(map(lambda x: read_raster(x, dtype='int8'), labels)))
labels = tile_sequences(labels, tile_size=(1, 1))
labels = np.squeeze(labels, axis=(2, 3)).astype(float)

# # Samples tiles (temporary), equal samples
# random.seed(1)
# index = np.concatenate([
#     random.choice(np.where(labels == 0)[0], 5000),
#     random.choice(np.where(labels == 3)[0], 5000)
# ])
# images = images[index]
# labels = labels[index]
# labels = np.where(labels == 3, 1.0, 0.0) # Converts to float
# del index

# Reshape for convolutional neural net
n, t, h, w, d = images.shape
images = images.reshape(n*t, h, w,d)


n, t, c = labels.shape
labels=labels.reshape(n*t, c)
labels = np.where(labels == 3, 1.0, 0.0) # Converts to float


# Splits samples (separate test sample)
np.random.seed(1)
samples_size= dict(train=0.8, valid=0.1, test=0.1)
index   = np.random.choice(np.arange(len(samples_size)) + 1, len(images), p=list(samples_size.values()))
images_train, images_valid, images_test = sample_split(images, index)
labels_train, labels_valid, labels_test = sample_split(labels, index)
del index


#%% AUGMENTATION

# Generator parameters
augmentation = dict(
    rescale=1./255,
    horizontal_flip=True, 
    vertical_flip=True,
    rotation_range=10, 
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.9,1.1],
    zoom_range=[0.9, 1.1],
    fill_mode='nearest'
)

# Initialises data generator
train_generator = preprocessing.image.ImageDataGenerator(**augmentation)
train_generator = train_generator.flow(images_train, labels_train, batch_size=32, shuffle=True, seed=1)
# del images_train, labels_train

#%% MODELS

# Initialises model
args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
model = models.convolutional_network(**args)
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics='accuracy')
# display_structure(model, path.join(paths['models'], 'convolutional_network.html'))


#%% OPTIMISATION

# Callbacks
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    callbacks.BackupAndRestore(backup_dir='../models')
]

# Estimates parameters
training = model.fit(
    train_generator,
#     steps_per_epoch=samples_size['train'] // 32,
    validation_data=(images_valid, labels_valid),
    epochs=100,
    verbose=1,
    callbacks=train_callbacks
)


Epoch 2/100


2022-06-03 10:08:31.232803: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


979/979 [==============================] - ETA: 0s - loss: 0.0111 - accuracy: 0.9958

2022-06-03 10:09:52.480399: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


979/979 [==============================] - 83s 83ms/step - loss: 0.0111 - accuracy: 0.9958 - val_loss: 1.2057 - val_accuracy: 0.9982
Epoch 3/100
979/979 [==============================] - 82s 84ms/step - loss: 0.0063 - accuracy: 0.9973 - val_loss: 1.0310 - val_accuracy: 0.9982
Epoch 4/100
979/979 [==============================] - 84s 85ms/step - loss: 0.0058 - accuracy: 0.9973 - val_loss: 2.7015 - val_accuracy: 0.9982
Epoch 5/100
979/979 [==============================] - 85s 87ms/step - loss: 0.0059 - accuracy: 0.9973 - val_loss: 1.7719 - val_accuracy: 0.9982
Epoch 6/100
979/979 [==============================] - 85s 87ms/step - loss: 0.0058 - accuracy: 0.9973 - val_loss: 1.7850 - val_accuracy: 0.9982
Epoch 7/100
979/979 [==============================] - 87s 88ms/step - loss: 0.0056 - accuracy: 0.9973 - val_loss: 1.2777 - val_accuracy: 0.9982
Epoch 8/100
979/979 [==============================] - 88s 90ms/step - loss: 0.0057 - accuracy: 0.9973 - val_loss: 0.2752 - val_accuracy: 0.99

In [6]:
labels.shape

(9776, 4, 1)